In [ ]:
from pathlib import Path
from IPython.display import Image, SVG, display
import getpass
import os
import re
import time

# Works when this notebook starts in notebooks/, the repository root, or Gadi.
candidates = [Path.cwd(), *Path.cwd().parents,
              Path(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))]
repo_root = next((path for path in candidates
                  if (path / "exercises/CMakeLists.txt").is_file()
                  and (path / "job-script").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the intro-to-cuda checkout.")
os.chdir(repo_root)
job_dir = repo_root / "job-script"
username = getpass.getuser()


def show_job_output(job_name, job_id):
    job_number = job_id.split(".")[0]
    log = job_dir / f"{job_name}.o{job_number}"
    if log.is_file():
        print(log.read_text())
    else:
        print(f"Output is not available yet: {log.name}. Check qstat, then rerun this cell.")


def outputs_ready(paths, submitted_at):
    if all(path.is_file() and path.stat().st_mtime >= submitted_at for path in paths):
        return True
    print("New results are not available yet. Wait for this job to finish, then rerun this cell.")
    return False

print(f"Repository: {repo_root}")

# Understanding Warps in CUDA


> **Overview**
>
> * **Time:** 30 min
> 
> 1. Learn about warps in CUDA programming.
> 1. Understand how warps affect performance and execution in CUDA kernels.

Warps are a fundamental concept in CUDA programming, representing how threads are organized and executed on NVIDIA GPUs. Understanding warps is crucial
for writing efficient CUDA kernels.

## What is a Warp?

A warp is a group of threads that execute the same instruction at the same time on a CUDA-enabled GPU.

* A **warp** consists of **32 threads** (in all current GPUs) that are executed **in parallel**.
* When a CUDA kernel is launched, threads are grouped into **blocks**, which are further divided into **warps** of 32 threads each.
* The 32 threads in a warp execute the **same instruction simultaneously** (SIMT - Single Instruction, Multiple Threads), but on **different data**.

## Why Use Warps?


Warps are designed to:

* Maximize **instruction-level parallelism**.
* Efficiently utilize the GPU's **SIMD-style execution units**.
* Enable lightweight and efficient scheduling using the **warp scheduler**.

## Thread Grouping into Warps


If a thread block contains 128 threads:

```text
128 threads / 32 threads per warp = 4 warps
```

* Threads 0–31 form warp 0.
* Threads 32–63 form warp 1.
* Threads 64–95 form warp 2.
* Threads 96–127 form warp 3.

## Execution Behavior


* **Threads in a warp execute in lockstep**, following the same instruction.
* If threads diverge in control flow (e.g., due to `if` conditions), **warp divergence** occurs, causing **serialized execution** and reduced performance.

## Warp Divergence


Warp divergence happens when threads in the same warp **follow different execution paths**.

Example:

```cpp
int tid = threadIdx.x;
if (tid % 2 == 0) {
    // Even threads do this
} else {
    // Odd threads do that
}
```

In this example, even and odd threads follow different branches, leading to warp divergence. This means that the warp will execute the even threads first,
then the odd threads, effectively serializing the execution and reducing performance.


Similarly it is a bad practice to have threads numbers that are not a multiple of 32 in a warp. This can lead to inefficient use of resources and increased
execution time.

For example, if a CUDA thread block has 30 threads, the last two threads in the warp will not be utilized, leading to wasted resources.
If a CUDA thread block has 40 threads, the block will be split as follows:

* Warp 0: threads 0–31
* Warp 1: threads 32–39 (only 8 threads)

So, the block will contain 2 warps, but the second warp is partially full (only 8 active threads).

* Warp 0 will execute normally with all 32 threads.
* Warp 1 will execute with only 8 active threads and the remaining 24 threads inactive (masked out).

The GPU still treats it as a full warp in terms of scheduling. Performance-wise, you're paying the cost of a full warp but using only part of it.


> **Important**
>
> * Write control flow to **minimize warp divergence**.
> * Ensure that threads access **contiguous memory** for coalesced memory access.
> * Aim for all threads in a warp to **follow the same path** for maximum efficiency.

## Some additional details


* Each SM is divided into processing partitions (also called sub-partitions).

* Each partition contains its own warp scheduler and instruction dispatch unit, along with a portion of the execution pipelines.

* On Volta (V100), an SM has 4 processing partitions, each with a warp scheduler, for a total of 4 schedulers per SM.

* An SM can host up to 64 resident warps (2048 threads).

* In each cycle, up to 4 warps can issue instructions (one per scheduler), while the rest wait.

* This deep pool of resident warps allows the GPU to quickly swap in ready warps when others are stalled (e.g., on memory or synchronization), ensuring high utilization and latency hiding.

In [ ]:
# Processing Partitions
display(Image(filename=str(repo_root / "docs/source/tutorial/figs/sm-partitions.png")))

> **Key Points**
>
> 1. A warp is a group of 32 threads that execute the same instruction simultaneously.
> 1. Warps are fundamental to CUDA's execution model, enabling parallel processing.
> 1. Warp divergence can lead to performance issues; aim to minimize it.
> 1. Threads should be organized to maximize coalesced memory access and minimize divergence.
> 1. Understanding warps is essential for writing efficient CUDA kernels and optimizing performance.

## Exercise 05: Warps

**Time: 20 min**

The job records timestamps for one block of 256 threads (eight warps).
[Exercise instructions](../docs/source/tutorial/exercises/05-warps.rst).

Run the setup cell first. Build the exercises once in
[02-gpu.ipynb](02-gpu.ipynb) before submitting jobs from this notebook.
After editing CUDA source, return to that notebook to rebuild.

Submission cells use the same `qsub` commands as the exercise documentation,
from `job-script`, and immediately show `qstat`. A submitted job may still be queued.
Use the monitoring cell to check again; rerunning a submission cell creates another job.
Read the PBS output and inspect generated files after the job finishes.

<a href="../exercises/05-warps/warp-timing.cu" target="_blank" rel="noopener noreferrer">Open warp-timing.cu</a>

In [ ]:
# Submit a new job, then inspect the queue.
submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 05-warps.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


### Check the queue again

Run this cell to monitor jobs without submitting another one. Once your job is no
longer listed, rerun its output/visualisation cells. Inspect the PBS log for errors.

In [ ]:
!qstat -u "{username}"


In [ ]:
!qstat -u "{username}"
show_job_output("05-warps", job_id)


### Plot the completed job's timestamps

The left panel shows which warp owns each thread. The right panel shows **when
each warp reached the clock read**: the dot marks its earliest thread reading,
and the horizontal line extends to its latest reading. If all 32 readings are
identical, the range collapses to one dot.

Offsets are relative to the earliest reading across the whole block, in SM clock
cycles. They are not execution durations. The plots use standard Python and
inline SVG; no plotting packages are needed.


In [ ]:
# Set True to plot saved timing data.
# Set False and rerun this cell to switch both plots back to live CSVs.
use_saved_warp_output = False


In [ ]:
import csv

csv_path = repo_root / "exercises/05-warps/timestamps.csv"
is_sample = globals().get("use_saved_warp_output", False)
if is_sample:
    log = (repo_root / "sample-outputs/05-warps.o184402113").read_text()
    readings = re.findall(r"^\d+,\d+,\d+,\d+$", log, re.MULTILINE)
    csv_lines = ["thread,warp,lane,clock", *readings]
elif "submitted_at" not in globals():
    print("Submit the GPU job or enable the collapsed sample cell.")
    csv_lines = None
elif outputs_ready([csv_path], submitted_at):
    csv_lines = csv_path.read_text().splitlines()
else:
    csv_lines = None

if csv_lines is not None:
    data = [{key: int(value) for key, value in row.items()}
            for row in csv.DictReader(csv_lines)]
    if not data:
        raise ValueError("No clock readings were found in the selected output.")
    if is_sample:
        print(f"Saved PBS excerpt: {len(data)} readings. A completed live run provides the full block.")
    assert is_sample or len(data) == 256
    assert [row["thread"] for row in data] == list(range(len(data)))
    assert all(row["warp"] == row["thread"] // 32 and row["lane"] == row["thread"] % 32
               for row in data)
    origin = min(row["clock"] for row in data)
    spread = max(row["clock"] for row in data) - origin
    palette = ["#dbeafe", "#ffedd5", "#dcfce7", "#f3e8ff", "#fce7f3", "#cffafe", "#e2e8f0", "#fef3c7"]
    parts = ['<svg xmlns="http://www.w3.org/2000/svg" width="1280" height="440" viewBox="0 0 1280 440">',
             '<rect width="1280" height="440" fill="white"/>',
             '<g font-family="monospace" fill="#172033">',
             '<text x="64" y="35" font-size="18">Warp membership</text>',
             '<text x="680" y="35" font-size="18">When each warp reached the clock read</text>']
    if is_sample:
        parts.append('<text x="90" y="58" font-size="13">Saved PBS excerpt: first 16 readings; offsets relative to this excerpt</text>')
    plotted_warps = sorted({row["warp"] for row in data})
    # Left: 32 threads per row, with row colours identifying their warp.
    for row in data:
        x, y = 64 + row["lane"] * 17, 85 + row["warp"] * 28
        parts.append(f'<rect x="{x}" y="{y}" width="17" height="28" fill="{palette[row["warp"]]}" stroke="#94a3b8" stroke-width="0.5"/>')
    for warp in plotted_warps:
        y = 99 + warp * 28
        parts.append(f'<text x="36" y="{y + 5}" font-size="14">{warp}</text>')
    for lane in [0, 7, 15, 23, 31]:
        parts.append(f'<text x="{64 + lane * 17}" y="337" font-size="12">{lane}</text>')
    parts.append('<text x="64" y="365" font-size="13">Rows: warp ID; columns: lane ID</text>')

    # Right: one dot at each warp's earliest offset, a range to its latest.
    axis_left, axis_right, axis_y = 730, 1210, 322
    axis_max = max(1, spread)  # Keep the axis valid if every timestamp is identical.
    def position(offset):
        return axis_left + 10 + offset / axis_max * (axis_right - axis_left - 20)
    tick_step = max(1, (axis_max + 4) // 5)
    ticks = sorted(set([*range(0, axis_max + 1, tick_step), axis_max]))
    for tick in ticks:
        x = position(tick)
        parts.append(f'<line x1="{x}" y1="76" x2="{x}" y2="{axis_y}" stroke="#e2e8f0"/>')
        parts.append(f'<text x="{x}" y="344" text-anchor="middle" font-size="12">{tick}</text>')
    parts.append(f'<line x1="{axis_left}" y1="{axis_y}" x2="{axis_right}" y2="{axis_y}" stroke="#172033"/>')
    print("Warp   earliest   latest   (SM clock cycles after first reading)")
    for warp in plotted_warps:
        offsets = [row["clock"] - origin for row in data if row["warp"] == warp]
        earliest, latest = min(offsets), max(offsets)
        x1, x2, y = position(earliest), position(latest), 99 + warp * 28
        parts.append(f'<g data-warp="{warp}" data-earliest="{earliest}" data-latest="{latest}">')
        parts.append(f'<text x="704" y="{y + 5}" font-size="14">{warp}</text>')
        parts.append(f'<line class="warp-range" x1="{x1}" y1="{y}" x2="{x2}" y2="{y}" stroke="#2563eb" stroke-width="3"/>')
        if latest != earliest:
            parts.append(f'<line x1="{x2}" y1="{y-5}" x2="{x2}" y2="{y+5}" stroke="#2563eb" stroke-width="2"/>')
        parts.append(f'<circle class="warp-start" cx="{x1}" cy="{y}" r="6" fill="{palette[warp]}" stroke="#172033" stroke-width="2"/>')
        parts.append('</g>')
        print(f"{warp:4d}   {earliest:8d}   {latest:6d}")
    parts.append('<text x="730" y="365" font-size="13">Timestamp offset (SM clock cycles); rows: warp ID</text>')
    parts.append('<text x="64" y="399" font-size="13">Dot = earliest reading; line = earliest to latest among the plotted threads in that warp.</text>')
    parts.append('<text x="64" y="423" font-size="13">Clock-read timing only: these ranges do not measure warp execution duration.</text>')
    parts.append('</g></svg>')
    svg = "".join(parts)
    plot_dir = repo_root / "notebook-output/05-warps"
    plot_dir.mkdir(parents=True, exist_ok=True)
    (plot_dir / ("warp-timing-sample.svg" if is_sample else "warp-timing.svg")).write_text(svg)
    display(SVG(data=svg))
    print(f"Timestamp offsets: 0 to {spread} SM clock cycles.")
    if spread == 0:
        print("All plotted threads recorded the same timestamp; all dots lie at zero.")

Repeat the job and compare the timestamp patterns. Does warp 0 always record
first? Why can threads from different warps have similar timestamps?

## Additional example: one block of 1,024 threads

Rebuild in `02-gpu.ipynb` before resubmitting. This program now launches **one block
with 1,024 threads**, forming **32 full warps**. Every thread reaches the barrier
and records one SM clock reading. The original 256-thread example is unchanged.

<a href="../exercises/05-warps/thread-timing.cu" target="_blank" rel="noopener noreferrer">Open thread-timing.cu</a>


In [ ]:
# Submit a new job, then inspect the queue.
thread_submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 05-thread-timing.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
thread_job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if thread_job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")

### Check the job output

Rerun this cell to check the queue without submitting another job. After completion,
read this job's log and run the per-thread plot below.

In [ ]:
!qstat -u "{username}"
show_job_output("05-thread-timing", thread_job_id)

### Thread timing line graph

The horizontal axis covers all 1,024 threads in one block. The vertical axis
shows each clock reading relative to the earliest available reading, in SM cycles.
A complete CSV supplies one point per thread. The archived PBS log supplies
threads 0–15, so the rest of the line is absent when using that excerpt.


In [ ]:
import csv

csv_path = repo_root / "exercises/05-warps/thread-timestamps.csv"
is_sample = globals().get("use_saved_warp_output", False)
sample_csv = repo_root / "sample-outputs/thread-timestamps.csv"
if is_sample and sample_csv.is_file():
    csv_lines = sample_csv.read_text().splitlines()
elif is_sample:
    log = (repo_root / "sample-outputs/05-thread-timing.o184402105").read_text()
    readings = re.findall(r"^\d+,\d+,\d+,\d+$", log, re.MULTILINE)
    csv_lines = ["thread,warp,lane,clock", *readings]
elif "thread_submitted_at" not in globals():
    print("Submit the GPU job or enable the collapsed sample cell.")
    csv_lines = None
elif outputs_ready([csv_path], thread_submitted_at):
    csv_lines = csv_path.read_text().splitlines()
else:
    csv_lines = None

if csv_lines is not None:
    thread_rows = [{key: int(value) for key, value in row.items()}
            for row in csv.DictReader(csv_lines)]
    if not thread_rows:
        raise ValueError("No clock readings were found in the selected output.")
    expected_threads = 1024
    assert len(thread_rows) <= expected_threads
    assert (is_sample and not sample_csv.is_file()) or len(thread_rows) == expected_threads
    assert [row["thread"] for row in thread_rows] == list(range(len(thread_rows)))
    assert all(row["warp"] == row["thread"] // 32 and row["lane"] == row["thread"] % 32
               for row in thread_rows)
    count = len(thread_rows)
    first_reading = min(row["clock"] for row in thread_rows)
    spread = max(row["clock"] for row in thread_rows) - first_reading
    scale = max(1, spread)
    width, height = 760, 280
    left, right, top, bottom = 78, 728, 70, 226

    def thread_x(index):
        return left + index / (expected_threads - 1) * (right - left)

    def clock_y(offset):
        return bottom - 8 - offset / scale * (bottom - top - 16)

    source = "Saved output" if is_sample else "Live output"
    subtitle = f"{source} • {count:,} of {expected_threads:,} readings"
    if count < expected_threads:
        subtitle += f" • threads {count:,}–{expected_threads - 1:,} unavailable"
    elements = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}" role="img">',
        '<title>Thread clock-reading offsets</title>',
        f'<desc>{subtitle}</desc>',
        '<rect width="100%" height="100%" fill="white"/>',
        '<g font-family="monospace" fill="#172033">',
        '<text x="78" y="28" font-size="17">Thread clock-reading offsets</text>',
        f'<text x="78" y="48" font-size="12">{subtitle}</text>',
    ]
    for tick in sorted({i * scale // 4 for i in range(5)}):
        y = clock_y(tick)
        elements.append(f'<line x1="{left}" x2="{right}" y1="{y}" y2="{y}" stroke="#e2e8f0"/>')
        elements.append(f'<text x="{left - 9}" y="{y + 4}" text-anchor="end" font-size="11">{tick}</text>')
    elements.append(f'<path d="M {left} {top} V {bottom} H {right}" fill="none" stroke="#94a3b8"/>')
    for tick in (0, 256, 512, 768, expected_threads - 1):
        x = thread_x(tick)
        elements.append(f'<text x="{x}" y="{bottom + 17}" text-anchor="middle" font-size="11">{tick:,}</text>')
    points = " ".join(
        f'{thread_x(row["thread"]):.2f},{clock_y(row["clock"] - first_reading):.2f}'
        for row in thread_rows
    )
    elements.append(f'<polyline points="{points}" fill="none" stroke="#2563eb" stroke-width="2" stroke-linejoin="round" stroke-linecap="round"/>')
    elements.append('<text x="403" y="265" text-anchor="middle" font-size="12">Thread index</text>')
    elements.append('<text transform="translate(19 148) rotate(-90)" text-anchor="middle" font-size="12">Offset (SM cycles)</text>')
    elements.append('</g></svg>')
    thread_svg = "".join(elements)
    thread_plot_dir = repo_root / "notebook-output/05-warps"
    thread_plot_dir.mkdir(parents=True, exist_ok=True)
    (thread_plot_dir / ("thread-timing-sample.svg" if is_sample else "thread-timing.svg")).write_text(thread_svg)
    display(SVG(data=thread_svg))


Compare the thread-level plot with the earlier warp-level plot. Do threads from
one warp share a timestamp? Do different warps form timing groups?
Repeat the job to see whether the ordering and differences persist.
